<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/summarize_web_content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai beautifulsoup4 requests

In [ ]:
from openai import OpenAI
import requests
from bs4 import BeautifulSoup

In [ ]:
from getpass import getpass

api_key = getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key: ··········


In [ ]:
client = OpenAI(
    api_key=api_key
)

In [ ]:
memory = []

In [ ]:
def fetch_website_content(url):

    try:

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers)

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove script/style tags
        for script in soup(["script", "style"]):
            script.extract()

        text = soup.get_text()

        # Clean text
        cleaned_text = " ".join(text.split())

        return cleaned_text[:5000]

    except Exception as e:
        return f"Error fetching website: {e}"

In [ ]:
def ai_agent(user_input):

    global memory

    # Save User Message
    memory.append({
        "role": "user",
        "content": user_input
    })

    # Default Prompt
    prompt = user_input

    # TOOL INVOCATION
    if "http" in user_input:

        url = user_input.split()[-1]

        website_text = fetch_website_content(url)

        prompt = f"""
        You are given website content below.

        Your task is to summarize it clearly.

        WEBSITE CONTENT:
        {website_text}
        """

    # System Message
    messages = [
        {
            "role": "system",
            "content": """
            You are a helpful AI assistant with memory.

            IMPORTANT:
            If website content is provided,
            summarize ONLY the provided content.

            Do not say:
            - 'I cannot access websites'
            - 'I do not have internet access'

            because the website content is already provided to you.
            """
        }
    ]

    # Add Memory
    messages.extend(memory)

    # Add Prompt
    messages.append({
        "role": "user",
        "content": prompt
    })

    # OpenAI Response
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    reply = response.choices[0].message.content

    # Save Assistant Reply
    memory.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
while True:

    user_input = input("\nYou: ")

    if user_input.lower() == "exit":
        print("Agent stopped.")
        break

    response = ai_agent(user_input)

    print("\nAgent:", response)


You: Hi

Agent: Hello! How can I assist you today?

You: Summarize this website: https://en.wikipedia.org/wiki/Artificial_intelligence

Agent: The website content provides an overview of artificial intelligence (AI) as a field, including its goals, techniques, applications, ethics, history, and philosophy. Key areas discussed include:

1. **Goals**: AI aims for reasoning, problem-solving, knowledge representation, learning, natural language processing, perception, and social intelligence among others.
   
2. **Techniques**: Various methods such as search and optimization, logic, probabilistic reasoning, machine learning, neural networks, and deep learning are utilized in AI.

3. **Applications**: AI is applied in diverse sectors such as healthcare, gaming, finance, military, and generative AI.

4. **Ethics**: Ethical concerns regarding AI include privacy, algorithmic bias, misinformation, environmental impacts, and the risks of technological unemployment.

5. **History and Philosophy*